In [30]:
import sys
import pandas as pd

print(sys.version)
print(pd.__version__)

3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
3.0.5


In [31]:
listings = pd.read_csv('../data/listings.csv')

# 1)

In [118]:
central = ['Södermalms', 'Östermalms', 'Norrmalms', 'Kungsholmens']
in_central = listings['neighbourhood_cleansed'].isin(central)
print(f'{in_central.sum()} {in_central.mean():.4f}')

2331 0.5096


In [120]:
private = listings['room_type'] == 'Private room'
central_private = (in_central & private) # listings[mask]['room_type'] == 'Private room'
print(f'{central_private.sum()/in_central.sum():.4f}')
non_central_private = (~in_central & private)
print(f'{(non_central_private).sum()/(~in_central).sum():.4f}')

0.1510
0.1939


The share of 'Private room' is higher outside the centre.

# 2)

In [49]:
entire_home = listings['room_type'] == 'Entire home/apt'
room_for_4 = listings['accommodates'] >= 4
print(f'{(entire_home & room_for_4).sum()/entire_home.sum():.4f}')
print(f'{((~entire_home) & room_for_4).sum()/(~entire_home).sum():.4f}')


0.5566
0.0934


This tells us: Listings that are listed as entire_home are far more likely to have accommodation for 4 or more people.
This doesn't tell us: What the answer to life, the universe and everything is.

# 3)

In [81]:
fifty = listings['number_of_reviews'] >= 50
score_5 = listings['review_scores_rating'] == 5
fifty_and_score_5 = listings[fifty]['review_scores_rating'] == 5
fifty_and_score_5_from_all = (fifty & score_5)
print(fifty_and_score_5.sum(), "of", fifty_and_score_5.count())

14 of 769


In [84]:
view = ['name', 'neighbourhood_cleansed', 'number_of_reviews', 'review_scores_rating']
listings[view][fifty_and_score_5_from_all].sort_values(
    ['review_scores_rating', 'number_of_reviews'], ascending=[False, False]).head()


,name,neighbourhood_cleansed,number_of_reviews,review_scores_rating
559,Breakfast paradise City Stockholm,Norrmalms,159,5.0
2115,"Creative, arty & colorful home. Feel like home!",Norrmalms,95,5.0
3566,Rum nära Vällingby Centrum,Hässelby-Vällingby,87,5.0
347,Stockholm,Norrmalms,77,5.0
531,Nice room located close to Globen,Enskede-Årsta-Vantörs,73,5.0


In [87]:
listings[fifty_and_score_5_from_all].nlargest(
    5, 'review_scores_rating')[view]


,name,neighbourhood_cleansed,number_of_reviews,review_scores_rating
97,"A one-of-a-kind apartment, overlooking the park",Norrmalms,59,5.0
169,"Historic home, nice garden close to Stockholm ...",Bromma,60,5.0
347,Stockholm,Norrmalms,77,5.0
513,House with a garden close to nature & city centre,Skarpnäcks,52,5.0
531,Nice room located close to Globen,Enskede-Årsta-Vantörs,73,5.0


The 2nd list is different, because we only sort by rating and not number of reviews (we could easily fix that)

# 4)

In [104]:
top = listings[view][fifty_and_score_5_from_all].sort_values(
    ['review_scores_rating', 'number_of_reviews'], ascending=[False, False]).head()

In [105]:
print(top.loc[347, 'name'])
print(top.iloc[3]['name'])

Stockholm
Stockholm


In [107]:
top = top.reset_index(drop=True)

In [108]:
print(top.loc[347], 'name')

KeyError: 347

the index of the 5 rows in top are 0, 1, 2, 3, 4 .... so loc[347] doesn't reference a valid row

In [110]:
print(top)

                                              name neighbourhood_cleansed  \
0                Breakfast paradise City Stockholm              Norrmalms   
1  Creative, arty & colorful home. Feel like home!              Norrmalms   
2                       Rum nära Vällingby Centrum     Hässelby-Vällingby   
3                                        Stockholm              Norrmalms   
4                Nice room located close to Globen  Enskede-Årsta-Vantörs   

   number_of_reviews  review_scores_rating  
0                159                   5.0  
1                 95                   5.0  
2                 87                   5.0  
3                 77                   5.0  
4                 73                   5.0  


# 5 a)

In [112]:
zero_bedrooms = listings['bedrooms'] == 0
print(zero_bedrooms.sum())
listings[zero_bedrooms]['room_type'].value_counts()

257


room_type
Entire home/apt    231
Private room        21
Shared room          5
Name: count, dtype: int64

In [114]:
print(listings[zero_bedrooms]['accommodates'].mean())

2.4785992217898833


most of them are listed as 'Entire home/apt' and low number of accommodates on average. So that tells us something, but not really enough to conclude exactly what is going on.

# b)

In [115]:
zero_avail_365 = listings['availability_365'] == 0
print(zero_avail_365.sum())

1494


In [116]:
zero_avail_365_and_no_reviews = listings[zero_avail_365]['number_of_reviews'] == 0
print(zero_avail_365_and_no_reviews.sum())

231


We can conclude that most of them have been reviewed, so most of them have been available at some point. We can't say for sure if they are fully booked for the next 365 days or if they are no longer available for booking. The most likely explaination seems that most of them have been available for booking in the past, but are no longer available for booking (seems more likely than the alternative, but that's just a guess, we can't know for sure).

# 6)

How many listings have availability_90 equal to 0 and how many of those have 0 reviews.
Does this give us a hint for the answer of 5b?

In [117]:
zero_avail_90 = listings['availability_90'] == 0
print(zero_avail_90.sum())
zero_avail_90_and_no_reviews = listings[zero_avail_90]['number_of_reviews'] == 0
print(zero_avail_90_and_no_reviews.sum())

1861
276
